# Logistics Data Science – Week 1
## Strategic Planning and Data Exploration

This notebook implements the proposed logistics analysis using the Olist Brazilian E-Commerce Public Dataset.

## 1. Business Problem

Understand delivery performance, freight cost, late-delivery risk and regional patterns in an e-commerce logistics network.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd().resolve().parent
sys.path.append(str(ROOT))

from src.data_cleaning import load_data, build_order_level_dataset
from src.kpi_analysis import calculate_kpis, monthly_kpis
from src.modelling import train_late_delivery_model, cluster_regions


## 2. Load the Data

Place the Olist CSV files in `data/` as described in `data/README.md`.

In [ ]:
orders, items, customers = load_data()
print("Orders:", orders.shape)
print("Items:", items.shape)
print("Customers:", customers.shape)


In [ ]:
df = build_order_level_dataset(orders, items, customers)
display(df.head())
display(df.isna().mean().sort_values(ascending=False).head(15))


## 3. Data Quality Checks

In [ ]:
print("Duplicate order IDs:", df["order_id"].duplicated().sum())
print("Status distribution:")
display(df["order_status"].value_counts(dropna=False))

invalid_dates = df[
    df["order_delivered_customer_date"].notna()
    & (df["order_delivered_customer_date"] < df["order_purchase_timestamp"])
]
print("Invalid purchase/delivery dates:", len(invalid_dates))


## 4. KPI Baseline

In [ ]:
kpis = calculate_kpis(df)
for name, value in kpis.items():
    print(f"{name}: {value}")


In [ ]:
monthly = monthly_kpis(df)
display(monthly.head())

monthly.plot(
    x="month",
    y="on_time_rate",
    marker="o",
    figsize=(12, 5)
)
plt.title("Monthly On-Time Delivery Rate")
plt.xlabel("Purchase Month")
plt.ylabel("On-Time Delivery Rate (%)")
plt.xticks(rotation=60)
plt.tight_layout()
plt.show()


## 5. Regional Exploration

In [ ]:
eligible = df[
    df["order_status"].eq("delivered") &
    df["delivery_days"].notna()
].copy()

regional = (
    eligible.groupby("customer_state")
    .agg(
        orders=("order_id", "nunique"),
        avg_delivery_days=("delivery_days", "mean"),
        late_rate=("is_late", "mean"),
        avg_freight=("freight_value", "mean")
    )
    .sort_values("late_rate", ascending=False)
)

regional["late_rate"] *= 100
display(regional.head(10))


In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=regional.reset_index(),
    x="avg_freight",
    y="late_rate",
    size="orders"
)
plt.title("Regional Freight Cost vs Late Delivery Rate")
plt.xlabel("Average Freight per Order")
plt.ylabel("Late Delivery Rate (%)")
plt.tight_layout()
plt.show()


## 6. Baseline Predictive Model

Logistic regression is used as an interpretable baseline for late-delivery classification.

In [ ]:
late_model = train_late_delivery_model(df)


## 7. Regional Clustering

K-Means identifies groups of states with similar delivery and freight characteristics.

In [ ]:
clusters = cluster_regions(df, n_clusters=4)
display(clusters.sort_values("cluster"))


## 8. Route Optimization – Strategic Extension

The next stage is to build a distance/time matrix from geographic coordinates and solve a constrained Vehicle Routing Problem using OR-Tools. Constraints may include vehicle capacity, route duration and customer time windows.

## 9. Business Questions

- Which regions have the highest late-delivery rates?
- Which segments combine high freight cost with poor service?
- Can late-delivery risk be predicted early enough for intervention?
- Which regional clusters require different strategies?
- How could optimized routes reduce distance or delivery time?

## 10. Conclusion

The workflow converts raw e-commerce logistics data into KPIs, exploratory insights, a baseline predictive model and regional segments. These outputs form the foundation for route optimization and operational recommendations.